In [1]:
# All Newman-alpha-scheme machinery (centering, Jacobians, LCF/spectral-gap
# computation, sync/async updates, MLE fitting, SBM data generation) now
# lives in utils.py -- see that file for full documentation of each function.
# utils.py must be in the same directory as this notebook (or on sys.path).
from utils import *

import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib.lines import Line2D

## Homogeneous

In [2]:
graph = 'balanced'   # SBM regime: 'balanced' (p=q, homogeneous),
                      # 'cluster' (p>>q, two well-separated communities),
                      # 'bipartite' (p=0-ish, q>0, near-bipartite graph)
n = 500               # number of objects/players in the comparison graph

# within-block (p) / cross-block (q) edge probabilities for the 2-block SBM
if graph == 'balanced':
    p = 0.05
    q = 0.05
elif graph == 'cluster':
    p = 0.1
    q = 0.01
elif graph == 'bipartite':
    p = 0.01
    q = 0.1

alpha_set = np.linspace(0, 1, 20)     # sweep of Newman's mixing parameter
A = sbm_2block(n, p, q, seed=1114)    # fixed comparison-graph topology for this SBM regime

# --- Shared plotting style config (kept local to each block for portability) ---
FIG_W, FIG_H  = 10.2, 9.5
TITLE_FS      = 42
LABEL_FS      = 40
TICK_FS       = 32
LEGEND_FS     = 30
MARKER_S      = 180
CI_ALPHA      = 0.15

COLOR_SYNC    = '#2166AC'   # blue -- both rho_sync and rho_bar_sync series
COLOR_ASYNC   = '#D6604D'   # red  -- both rho_async and rho_bar_async series

rc('text', usetex=True)
rc('font', family='serif')
rc('axes', linewidth=0.8)

L_set      = [10, 100]      # number of simulated comparisons per edge (small vs. large sample)
sigma_list = [0.1, 0.5]     # log-normal dispersion of the true strengths gamma_true
                            # (small sigma = near-homogeneous strengths, large sigma = high dynamic range)

for i_sigma, sigma in enumerate(sigma_list):
    # --- generate true strengths for this dynamic-range setting ---
    rng        = np.random.default_rng(365)
    gamma_true = rng.lognormal(0, sigma, n)
    gamma_true = centering(gamma_true)   # normalize geometric mean to 1
    print(f'Dynamic range: {gamma_true.max()/gamma_true.min():.4f}')

    for L in L_set:
        # --- simulate comparison data on graph A and fit the MLE ---
        np.random.seed(1023)
        W, win_list, loss_list, mle = get_data(A, L, gamma_true)
        alpha_set = np.linspace(0, 1, 20)

        # rates_*  = EMPIRICAL local convergence factor rho(alpha), computed
        #            via the exact Jacobian eigenvalues at the FITTED mle.
        # gaps_*   = THEORETICAL spectral gap, computed at the TRUE strengths
        #            gamma_true (not mle) -- this is the quantity the theory
        #            predicts should track rho(alpha) as n -> infty;
        #            rho_bar = 1 - gap.
        rates_f = np.array([get_lcf(W, mle, alpha, sync='full') for alpha in alpha_set])
        gaps_f  = np.array([get_gap(W, gamma_true, alpha, sync='full')  for alpha in alpha_set])

        rates_a = np.array([get_lcf(W, mle, alpha, sync='none') for alpha in alpha_set])
        gaps_a  = np.array([get_gap(W, gamma_true, alpha, sync='none')  for alpha in alpha_set])

        # ---------- summary print: empirical rho vs. theoretical rho_bar at alpha=0,1 ----------
        # (alpha_set = linspace(0, 1, 20), so index 0 <-> alpha=0 and index -1 <-> alpha=1)
        print('-------------------------------------------------------------------')
        print(f'sigma={sigma}, L={L}')
        print("Sync:")
        print(f"  rho(0)     = {rates_f[0]:.6f},  rho_bar(0) = {1 - gaps_f[0]:.6f}")
        print(f"  rho(1)     = {rates_f[-1]:.6f}, rho_bar(1) = {1 - gaps_f[-1]:.6f}")
        print("Async:")
        print(f"  rho(0)     = {rates_a[0]:.6f},  rho_bar(0) = {1 - gaps_a[0]:.6f}")
        print(f"  rho(1)     = {rates_a[-1]:.6f}, rho_bar(1) = {1 - gaps_a[-1]:.6f}")

        # ---------- DataFrame: log10-rate + "improvement" ratio vs. sync-alpha=1 baseline ----------
        # "improvement" = (rate_ratio, gap_pred_ratio) relative to the slowest
        # baseline case (sync, alpha=1), i.e. how much faster each case is on
        # a log10-error-per-iteration basis.
        rows = [
            {
                "case": "slope-a-0",
                "slope": np.log10(rates_a)[0],
                "gap_prediction": np.log10(1 - gaps_a)[0],
                "improvement": (np.log10(rates_a)[0]  / np.log10(rates_f)[-1],
                                np.log10(1 - gaps_a)[0]  / np.log10(1 - gaps_f)[-1]),
            },
            {
                "case": "slope-a-1",
                "slope": np.log10(rates_a)[-1],
                "gap_prediction": np.log10(1 - gaps_a)[-1],
                "improvement": (np.log10(rates_a)[-1] / np.log10(rates_f)[-1],
                                np.log10(1 - gaps_a)[-1] / np.log10(1 - gaps_f)[-1]),
            },
            {
                "case": "slope-s-0",
                "slope": np.log10(rates_f)[0],
                "gap_prediction": np.log10(1 - gaps_f)[0],
                "improvement": (np.log10(rates_f)[0]  / np.log10(rates_f)[-1],
                                np.log10(1 - gaps_f)[0]  / np.log10(1 - gaps_f)[-1]),
            },
            {
                "case": "slope-s-1",
                "slope": np.log10(rates_f)[-1],
                "gap_prediction": np.log10(1 - gaps_f)[-1],
                "improvement": (1.0, 1.0),   # baseline case, always ratio 1
            },
        ]
        # df = pd.DataFrame(rows)
        # df.to_csv(f"{graph}_{sigma*10}_{L}.csv", index=False)

        # ---------- plot: rho and rho_bar vs. alpha, sync (blue) and async (red) ----------
        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

        # Empirical rho_sync: hollow squares + solid line
        ax.scatter(alpha_set, rates_f, label=r'$\rho_{\mathrm{sync}}$',
                color=COLOR_SYNC,  s=MARKER_S,   marker='s', zorder=3, alpha=0.8, facecolors='none')
        ax.plot(alpha_set, rates_f, color=COLOR_SYNC, linewidth=1, zorder=2)

        # Theoretical rho_bar_sync: plus markers + dashed line
        ax.scatter(alpha_set, 1 - gaps_f, label=r'$\bar\rho_{\mathrm{sync}}$',
                color=COLOR_SYNC,  s=MARKER_S*2 + 40, marker='+', linewidths=1.5, zorder=3)
        ax.plot(alpha_set, 1 - gaps_f, color=COLOR_SYNC, linewidth=1, linestyle='--', zorder=2)

        # Empirical rho_async: hollow squares + solid line (red)
        ax.scatter(alpha_set, rates_a, label=r'$\rho_{\mathrm{async}}$',
                color=COLOR_ASYNC, s=MARKER_S,   marker='s', zorder=3, alpha=0.8, facecolors='none')
        ax.plot(alpha_set, rates_a, color=COLOR_ASYNC, linewidth=1, zorder=2)

        # Theoretical rho_bar_async: plus markers + dashed line (red)
        ax.scatter(alpha_set, 1 - gaps_a, label=r'$\bar\rho_{\mathrm{async}}$',
                color=COLOR_ASYNC, s=MARKER_S*2, marker='+', linewidths=1.5, zorder=3)
        ax.plot(alpha_set, 1 - gaps_a, color=COLOR_ASYNC, linewidth=1, linestyle='--', zorder=2)

        ax.set_title(rf"$\sigma={sigma},\ L={L}$", fontsize=TITLE_FS, pad=20)
        ax.set_xlabel(r'$\alpha$', fontsize=LABEL_FS)
        # --- Referee comment: unify y-axis label from 'Rate' to 'Convergence factor' ---
        ax.set_ylabel(r'Convergence factor', fontsize=LABEL_FS, labelpad=20)
        ax.set_ylim(-0.05, 1.05)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FS)
        ax.grid(True, linewidth=0.4, alpha=0.4)

        # Custom legend handles (drawn once, reused across sigma/L combos)
        legend_elements = [
            Line2D([0,1], [0,0], color=COLOR_SYNC,  marker='s', markersize=10,
                markerfacecolor='none', markeredgecolor=COLOR_SYNC,
                linestyle='-',  linewidth=1, label=r'$\rho_{\mathrm{sync}}$'),
            Line2D([0,1], [0,0], color=COLOR_SYNC,  marker='+', markersize=12,
                markeredgewidth=1.5,
                linestyle='--', linewidth=1, label=r'$\bar\rho_{\mathrm{sync}}$'),
            Line2D([0,1], [0,0], color=COLOR_ASYNC, marker='s', markersize=10,
                markerfacecolor='none', markeredgecolor=COLOR_ASYNC,
                linestyle='-',  linewidth=1, label=r'$\rho_{\mathrm{async}}$'),
            Line2D([0,1], [0,0], color=COLOR_ASYNC, marker='+', markersize=12,
                markeredgewidth=1.5,
                linestyle='--', linewidth=1, label=r'$\bar\rho_{\mathrm{async}}$'),
        ]

        # Only draw the legend once (on the first panel), to avoid clutter
        # across the full sigma x L grid of figures.
        if sigma == 0.1 and L == 10 and graph == 'balanced':
            ax.legend(handles=legend_elements, fontsize=LEGEND_FS, loc='upper left',
                    framealpha=0.85, edgecolor='gray')

        plt.tight_layout(pad=0.5)
        plt.savefig(f"{graph}_{sigma*10}_{L}.pdf", bbox_inches='tight', dpi=300)
        plt.close()

Dynamic range: 1.8988
-------------------------------------------------------------------
sigma=0.1, L=10
Sync:
  rho(0)     = 0.375103,  rho_bar(0) = 0.375073
  rho(1)     = 0.699669, rho_bar(1) = 0.691288
Async:
  rho(0)     = 0.148047,  rho_bar(0) = 0.148980
  rho(1)     = 0.670652, rho_bar(1) = 0.658424
-------------------------------------------------------------------
sigma=0.1, L=100
Sync:
  rho(0)     = 0.375111,  rho_bar(0) = 0.375073
  rho(1)     = 0.692166, rho_bar(1) = 0.691288
Async:
  rho(0)     = 0.148959,  rho_bar(0) = 0.148980
  rho(1)     = 0.659428, rho_bar(1) = 0.658424
Dynamic range: 24.6854
-------------------------------------------------------------------
sigma=0.5, L=10
Sync:
  rho(0)     = 0.377298,  rho_bar(0) = 0.375803
  rho(1)     = 0.847692, rho_bar(1) = 0.854235
Async:
  rho(0)     = 0.151399,  rho_bar(0) = 0.148186
  rho(1)     = 0.846307, rho_bar(1) = 0.853034
-------------------------------------------------------------------
sigma=0.5, L=100
Sync:
  

## Bipartite

In [3]:
graph = 'bipartite'   # near-bipartite SBM regime: p << q
n = 500

if graph == 'balanced':
    p = 0.05
    q = 0.05
elif graph == 'cluster':
    p = 0.1
    q = 0.01
elif graph == 'bipartite':
    p = 0.01
    q = 0.1

alpha_set = np.linspace(0, 1, 20)
A = sbm_2block(n, p, q, seed=2026)   # note: different seed than the Homogeneous block

# --- (styling and the sigma/L double loop below are IDENTICAL in structure
#      to the Homogeneous block above -- see those comments for details on
#      what rates_f/gaps_f/rates_a/gaps_a represent, the CSV export format,
#      and the plotting/legend logic.) ---

FIG_W, FIG_H  = 10.2, 9.5
TITLE_FS      = 42
LABEL_FS      = 40
TICK_FS       = 32
LEGEND_FS     = 30
MARKER_S      = 180
CI_ALPHA      = 0.15

COLOR_SYNC    = '#2166AC'
COLOR_ASYNC   = '#D6604D'

rc('text', usetex=True)
rc('font', family='serif')
rc('axes', linewidth=0.8)

L_set      = [10, 100]
sigma_list = [0.1, 0.5]

for i_sigma, sigma in enumerate(sigma_list):
    rng        = np.random.default_rng(365)
    gamma_true = rng.lognormal(0, sigma, n)
    gamma_true = centering(gamma_true)
    print(f'Dynamic range: {gamma_true.max()/gamma_true.min():.4f}')

    for L in L_set:
        np.random.seed(1023)
        W, win_list, loss_list, mle = get_data(A, L, gamma_true)
        alpha_set = np.linspace(0, 1, 20)

        rates_f = np.array([get_lcf(W, mle, alpha, sync='full') for alpha in alpha_set])
        gaps_f  = np.array([get_gap(W, gamma_true, alpha, sync='full')  for alpha in alpha_set])
        rates_a = np.array([get_lcf(W, mle, alpha, sync='none') for alpha in alpha_set])
        gaps_a  = np.array([get_gap(W, gamma_true, alpha, sync='none')  for alpha in alpha_set])

        print('-------------------------------------------------------------------')
        print(f'sigma={sigma}, L={L}')
        print("Sync:")
        print(f"  rho(0)     = {rates_f[0]:.6f},  rho_bar(0) = {1 - gaps_f[0]:.6f}")
        print(f"  rho(1)     = {rates_f[-1]:.6f}, rho_bar(1) = {1 - gaps_f[-1]:.6f}")
        print("Async:")
        print(f"  rho(0)     = {rates_a[0]:.6f},  rho_bar(0) = {1 - gaps_a[0]:.6f}")
        print(f"  rho(1)     = {rates_a[-1]:.6f}, rho_bar(1) = {1 - gaps_a[-1]:.6f}")

        rows = [
            {"case": "slope-a-0", "slope": np.log10(rates_a)[0],
             "gap_prediction": np.log10(1 - gaps_a)[0],
             "improvement": (np.log10(rates_a)[0]/np.log10(rates_f)[-1],
                             np.log10(1-gaps_a)[0]/np.log10(1-gaps_f)[-1])},
            {"case": "slope-a-1", "slope": np.log10(rates_a)[-1],
             "gap_prediction": np.log10(1 - gaps_a)[-1],
             "improvement": (np.log10(rates_a)[-1]/np.log10(rates_f)[-1],
                             np.log10(1-gaps_a)[-1]/np.log10(1-gaps_f)[-1])},
            {"case": "slope-s-0", "slope": np.log10(rates_f)[0],
             "gap_prediction": np.log10(1 - gaps_f)[0],
             "improvement": (np.log10(rates_f)[0]/np.log10(rates_f)[-1],
                             np.log10(1-gaps_f)[0]/np.log10(1-gaps_f)[-1])},
            {"case": "slope-s-1", "slope": np.log10(rates_f)[-1],
             "gap_prediction": np.log10(1 - gaps_f)[-1],
             "improvement": (1.0, 1.0)},
        ]
        # df = pd.DataFrame(rows)
        # df.to_csv(f"{graph}_{sigma*10}_{L}.csv", index=False)

        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        ax.scatter(alpha_set, rates_f, label=r'$\rho_{\mathrm{sync}}$',
                color=COLOR_SYNC,  s=MARKER_S,   marker='s', zorder=3, alpha=0.8, facecolors='none')
        ax.plot(alpha_set, rates_f, color=COLOR_SYNC, linewidth=1, zorder=2)
        ax.scatter(alpha_set, 1 - gaps_f, label=r'$\bar\rho_{\mathrm{sync}}$',
                color=COLOR_SYNC,  s=MARKER_S*2 + 40, marker='+', linewidths=1.5, zorder=3)
        ax.plot(alpha_set, 1 - gaps_f, color=COLOR_SYNC, linewidth=1, linestyle='--', zorder=2)
        ax.scatter(alpha_set, rates_a, label=r'$\rho_{\mathrm{async}}$',
                color=COLOR_ASYNC, s=MARKER_S,   marker='s', zorder=3, alpha=0.8, facecolors='none')
        ax.plot(alpha_set, rates_a, color=COLOR_ASYNC, linewidth=1, zorder=2)
        ax.scatter(alpha_set, 1 - gaps_a, label=r'$\bar\rho_{\mathrm{async}}$',
                color=COLOR_ASYNC, s=MARKER_S*2, marker='+', linewidths=1.5, zorder=3)
        ax.plot(alpha_set, 1 - gaps_a, color=COLOR_ASYNC, linewidth=1, linestyle='--', zorder=2)

        ax.set_title(rf"$\sigma={sigma},\ L={L}$", fontsize=TITLE_FS, pad=20)
        ax.set_xlabel(r'$\alpha$', fontsize=LABEL_FS)
        # --- Referee comment: unify y-axis label from 'Rate' to 'Convergence factor' ---
        ax.set_ylabel(r'Convergence factor', fontsize=LABEL_FS, labelpad=20)
        ax.set_ylim(-0.05, 1.05)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FS)
        ax.grid(True, linewidth=0.4, alpha=0.4)

        # NOTE: this legend condition checks `graph == 'balanced'`, which is
        # never true inside this bipartite block -- so no legend is ever
        # drawn here (kept as-is from the original code; the legend is only
        # meant to appear once, on the Homogeneous figure).
        if sigma == 0.1 and L == 10 and graph == 'balanced':
            ax.legend(fontsize=LEGEND_FS, loc='upper left',
                      framealpha=0.85, edgecolor='gray')

        plt.tight_layout(pad=0.5)
        plt.savefig(f"{graph}_{sigma*10}_{L}.pdf", bbox_inches='tight', dpi=300)
        plt.close()

Dynamic range: 1.8988
-------------------------------------------------------------------
sigma=0.1, L=10
Sync:
  rho(0)     = 0.828951,  rho_bar(0) = 0.828792
  rho(1)     = 0.688374, rho_bar(1) = 0.681501
Async:
  rho(0)     = 0.130121,  rho_bar(0) = 0.128291
  rho(1)     = 0.659432, rho_bar(1) = 0.648604
-------------------------------------------------------------------
sigma=0.1, L=100
Sync:
  rho(0)     = 0.828826,  rho_bar(0) = 0.828792
  rho(1)     = 0.682641, rho_bar(1) = 0.681501
Async:
  rho(0)     = 0.128383,  rho_bar(0) = 0.128291
  rho(1)     = 0.650254, rho_bar(1) = 0.648604
Dynamic range: 24.6854
-------------------------------------------------------------------
sigma=0.5, L=10
Sync:
  rho(0)     = 0.829171,  rho_bar(0) = 0.828748
  rho(1)     = 0.851769, rho_bar(1) = 0.843714
Async:
  rho(0)     = 0.135872,  rho_bar(0) = 0.128261
  rho(1)     = 0.850659, rho_bar(1) = 0.842409
-------------------------------------------------------------------
sigma=0.5, L=100
Sync:
  

## Cluster

In [4]:
graph = 'cluster'   # clustered SBM regime: p >> q (two communities)
n = 500

if graph == 'balanced':
    p = 0.05
    q = 0.05
elif graph == 'cluster':
    p = 0.1
    q = 0.01
elif graph == 'bipartite':
    p = 0.01
    q = 0.1

alpha_set = np.linspace(0, 1, 20)
A = sbm_2block(n, p, q, seed=2026)

# --- (styling and the sigma/L double loop below are IDENTICAL in structure
#      to the Homogeneous block above.) ---

FIG_W, FIG_H  = 10.2, 9.5
TITLE_FS      = 42
LABEL_FS      = 40
TICK_FS       = 32
LEGEND_FS     = 30
MARKER_S      = 180
CI_ALPHA      = 0.15

COLOR_SYNC    = '#2166AC'
COLOR_ASYNC   = '#D6604D'

rc('text', usetex=True)
rc('font', family='serif')
rc('axes', linewidth=0.8)

L_set      = [10, 100]
sigma_list = [0.1, 0.5]

for i_sigma, sigma in enumerate(sigma_list):
    rng        = np.random.default_rng(365)
    gamma_true = rng.lognormal(0, sigma, n)
    gamma_true = centering(gamma_true)
    print(f'Dynamic range: {gamma_true.max()/gamma_true.min():.4f}')

    for L in L_set:
        np.random.seed(1023)
        W, win_list, loss_list, mle = get_data(A, L, gamma_true)
        alpha_set = np.linspace(0, 1, 20)

        rates_f = np.array([get_lcf(W, mle, alpha, sync='full') for alpha in alpha_set])
        gaps_f  = np.array([get_gap(W, gamma_true, alpha, sync='full')  for alpha in alpha_set])
        rates_a = np.array([get_lcf(W, mle, alpha, sync='none') for alpha in alpha_set])
        gaps_a  = np.array([get_gap(W, gamma_true, alpha, sync='none')  for alpha in alpha_set])

        print('-------------------------------------------------------------------')
        print(f'sigma={sigma}, L={L}')
        print("Sync:")
        print(f"  rho(0)     = {rates_f[0]:.6f},  rho_bar(0) = {1 - gaps_f[0]:.6f}")
        print(f"  rho(1)     = {rates_f[-1]:.6f}, rho_bar(1) = {1 - gaps_f[-1]:.6f}")
        print("Async:")
        print(f"  rho(0)     = {rates_a[0]:.6f},  rho_bar(0) = {1 - gaps_a[0]:.6f}")
        print(f"  rho(1)     = {rates_a[-1]:.6f}, rho_bar(1) = {1 - gaps_a[-1]:.6f}")

        rows = [
            {"case": "slope-a-0", "slope": np.log10(rates_a)[0],
             "gap_prediction": np.log10(1 - gaps_a)[0],
             "improvement": (np.log10(rates_a)[0]/np.log10(rates_f)[-1],
                             np.log10(1-gaps_a)[0]/np.log10(1-gaps_f)[-1])},
            {"case": "slope-a-1", "slope": np.log10(rates_a)[-1],
             "gap_prediction": np.log10(1 - gaps_a)[-1],
             "improvement": (np.log10(rates_a)[-1]/np.log10(rates_f)[-1],
                             np.log10(1-gaps_a)[-1]/np.log10(1-gaps_f)[-1])},
            {"case": "slope-s-0", "slope": np.log10(rates_f)[0],
             "gap_prediction": np.log10(1 - gaps_f)[0],
             "improvement": (np.log10(rates_f)[0]/np.log10(rates_f)[-1],
                             np.log10(1-gaps_f)[0]/np.log10(1-gaps_f)[-1])},
            {"case": "slope-s-1", "slope": np.log10(rates_f)[-1],
             "gap_prediction": np.log10(1 - gaps_f)[-1],
             "improvement": (1.0, 1.0)},
        ]
        # df = pd.DataFrame(rows)
        # df.to_csv(f"{graph}_{sigma*10}_{L}.csv", index=False)

        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        ax.scatter(alpha_set, rates_f, label=r'$\rho_{\mathrm{sync}}$',
                color=COLOR_SYNC,  s=MARKER_S,   marker='s', zorder=3, alpha=0.8, facecolors='none')
        ax.plot(alpha_set, rates_f, color=COLOR_SYNC, linewidth=1, zorder=2)
        ax.scatter(alpha_set, 1 - gaps_f, label=r'$\bar\rho_{\mathrm{sync}}$',
                color=COLOR_SYNC,  s=MARKER_S*2 + 40, marker='+', linewidths=1.5, zorder=3)
        ax.plot(alpha_set, 1 - gaps_f, color=COLOR_SYNC, linewidth=1, linestyle='--', zorder=2)
        ax.scatter(alpha_set, rates_a, label=r'$\rho_{\mathrm{async}}$',
                color=COLOR_ASYNC, s=MARKER_S,   marker='s', zorder=3, alpha=0.8, facecolors='none')
        ax.plot(alpha_set, rates_a, color=COLOR_ASYNC, linewidth=1, zorder=2)
        ax.scatter(alpha_set, 1 - gaps_a, label=r'$\bar\rho_{\mathrm{async}}$',
                color=COLOR_ASYNC, s=MARKER_S*2, marker='+', linewidths=1.5, zorder=3)
        ax.plot(alpha_set, 1 - gaps_a, color=COLOR_ASYNC, linewidth=1, linestyle='--', zorder=2)

        ax.set_title(rf"$\sigma={sigma},\ L={L}$", fontsize=TITLE_FS, pad=20)
        ax.set_xlabel(r'$\alpha$', fontsize=LABEL_FS)
        # --- Referee comment: unify y-axis label from 'Rate' to 'Convergence factor' ---
        ax.set_ylabel(r'Convergence factor', fontsize=LABEL_FS, labelpad=20)
        ax.set_ylim(-0.05, 1.05)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FS)
        ax.grid(True, linewidth=0.4, alpha=0.4)

        if sigma == 0.1 and L == 10 and graph == 'balanced':
            ax.legend(fontsize=LEGEND_FS, loc='upper left',
                      framealpha=0.85, edgecolor='gray')

        plt.tight_layout(pad=0.5)
        plt.savefig(f"{graph}_{sigma*10}_{L}.pdf", bbox_inches='tight', dpi=300)
        plt.close()

Dynamic range: 1.8988
-------------------------------------------------------------------
sigma=0.1, L=10
Sync:
  rho(0)     = 0.837128,  rho_bar(0) = 0.837028
  rho(1)     = 0.919569, rho_bar(1) = 0.918857
Async:
  rho(0)     = 0.706572,  rho_bar(0) = 0.706375
  rho(1)     = 0.898148, rho_bar(1) = 0.896981
-------------------------------------------------------------------
sigma=0.1, L=100
Sync:
  rho(0)     = 0.837024,  rho_bar(0) = 0.837028
  rho(1)     = 0.918955, rho_bar(1) = 0.918857
Async:
  rho(0)     = 0.706369,  rho_bar(0) = 0.706375
  rho(1)     = 0.897144, rho_bar(1) = 0.896981
Dynamic range: 24.6854
-------------------------------------------------------------------
sigma=0.5, L=10
Sync:
  rho(0)     = 0.837460,  rho_bar(0) = 0.837474
  rho(1)     = 0.927576, rho_bar(1) = 0.926859
Async:
  rho(0)     = 0.707158,  rho_bar(0) = 0.707087
  rho(1)     = 0.911144, rho_bar(1) = 0.909985
-------------------------------------------------------------------
sigma=0.5, L=100
Sync:
  

## Eigenvalue diagnostics (second-largest / smallest of J at the true strengths)

In [5]:
# ---------------------------------------------------------------------------
# Diagnostic: average of the SECOND-LARGEST and SMALLEST eigenvalues of the
# Jacobian J = get_jacobian(W, gamma_true, alpha=0), evaluated at the TRUE
# strengths gamma_true (not the fitted mle).
#
# IMPORTANT CAVEAT: get_jacobian's Jacobian is not symmetric in general, so
# np.linalg.eigvals(J) can return complex eigenvalues. This code assumes they
# are (numerically) real and discards the imaginary part via `.real` -- this
# is only valid because alpha=0 and this particular graph regime happen to
# produce a real spectrum; for other alpha or graph structures, silently
# dropping the imaginary part could hide a genuinely complex eigenvalue and
# give a misleading "average". If reusing this snippet elsewhere, check
# np.max(np.abs(np.linalg.eigvals(J).imag)) is negligible before dropping it.
# ---------------------------------------------------------------------------

graph = 'balanced'
n = 500

if graph == 'balanced':
    p = 0.05
    q = 0.05
elif graph == 'cluster':
    p = 0.1
    q = 0.01
elif graph == 'bipartite':
    p = 0.01
    q = 0.1

alpha_set = np.linspace(0, 1, 20)
A = sbm_2block(n, p, q, seed=1114)   # seed=1114 for 'balanced' (matches the rate-sweep cell above)

L_set      = [10, 100]
sigma_list = [0.1, 0.5]

for i_sigma, sigma in enumerate(sigma_list):
    rng        = np.random.default_rng(365)
    gamma_true = rng.lognormal(0, sigma, n)
    gamma_true = centering(gamma_true)
    for L in L_set:
        np.random.seed(1023)
        W, win_list, loss_list, mle = get_data(A, L, gamma_true)

        # Jacobian at the TRUE strengths (theoretical quantity), alpha=0
        J = get_jacobian(W, gamma_true, 0, full=False)
        evals = np.linalg.eigvals(J).real   # see caveat above
        evals_sorted = np.sort(evals)[::-1]  # descending order

        second_largest = evals_sorted[1]   # excludes the trivial eigenvalue 1
        smallest       = evals_sorted[-1]
        average        = (second_largest + smallest) / 2

        print(f"Average:                   {average:.6f}")

Average:                   0.001693
Average:                   0.001693
Average:                   0.000559
Average:                   0.000559


In [6]:
# Same diagnostic as above, for the CLUSTER regime (graph='cluster', seed=2026)
graph = 'cluster'
n = 500

if graph == 'balanced':
    p = 0.05
    q = 0.05
elif graph == 'cluster':
    p = 0.1
    q = 0.01
elif graph == 'bipartite':
    p = 0.01
    q = 0.1

alpha_set = np.linspace(0, 1, 20)
A = sbm_2block(n, p, q, seed=2026)

L_set      = [10, 100]
sigma_list = [0.1, 0.5]

for i_sigma, sigma in enumerate(sigma_list):
    rng        = np.random.default_rng(365)
    gamma_true = rng.lognormal(0, sigma, n)
    gamma_true = centering(gamma_true)
    for L in L_set:
        np.random.seed(1023)
        W, win_list, loss_list, mle = get_data(A, L, gamma_true)
        J = get_jacobian(W, gamma_true, 0, full=False)
        evals = np.linalg.eigvals(J).real
        evals_sorted = np.sort(evals)[::-1]

        second_largest = evals_sorted[1]
        smallest       = evals_sorted[-1]
        average        = (second_largest + smallest) / 2

        print(f"Average:                   {average:.6f}")

Average:                   0.237497
Average:                   0.237497
Average:                   0.237537
Average:                   0.237537


In [7]:
# Same diagnostic as above, for the BIPARTITE regime (graph='bipartite', seed=2026)
graph = 'bipartite'
n = 500

if graph == 'balanced':
    p = 0.05
    q = 0.05
elif graph == 'cluster':
    p = 0.1
    q = 0.01
elif graph == 'bipartite':
    p = 0.01
    q = 0.1

alpha_set = np.linspace(0, 1, 20)
A = sbm_2block(n, p, q, seed=2026)

L_set      = [10, 100]
sigma_list = [0.1, 0.5]

for i_sigma, sigma in enumerate(sigma_list):
    rng        = np.random.default_rng(365)
    gamma_true = rng.lognormal(0, sigma, n)
    gamma_true = centering(gamma_true)
    for L in L_set:
        np.random.seed(1023)
        W, win_list, loss_list, mle = get_data(A, L, gamma_true)
        J = get_jacobian(W, gamma_true, 0, full=False)
        evals = np.linalg.eigvals(J).real
        evals_sorted = np.sort(evals)[::-1]

        second_largest = evals_sorted[1]
        smallest       = evals_sorted[-1]
        average        = (second_largest + smallest) / 2

        print(f"Average:                   {average:.6f}")

Average:                   -0.237662
Average:                   -0.237662
Average:                   -0.236778
Average:                   -0.236778
